# Pass 2 — Consolidate & Deduplicate Objectives

**Input:** Pass 1 output (per-column extractions for each fund).  
**Task:** The LLM sees ALL per-column extractions for one fund and:  
1. Matches equivalent objectives across languages/columns  
2. Deduplicates (same objective stated in English, French, German = one objective)  
3. Produces a final consolidated list with English text and classification  

**Output:** One row per fund with final deduplicated objectives.

In [1]:
import pandas as pd
from tqdm import tqdm
import time, os, json, anthropic
from pathlib import Path

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

OUTPUT_DIR = Path(config["Output"])
MODEL = "claude-sonnet-4-6"  # UPDATE as needed

In [2]:
# === LOAD PASS 1 OUTPUT ===
# UPDATE this path to your actual Pass 1 output file
PASS1_FILE = os.path.join(OUTPUT_DIR, "Pass1_Extract_100_funds_20260517_2103.xlsx")  # UPDATE

p1_df = pd.read_excel(PASS1_FILE)
p1_df['pass1_raw'] = p1_df['pass1_raw'].apply(json.loads)
p1_df['columns_sent'] = p1_df['columns_sent'].apply(json.loads)
print(f"Loaded {len(p1_df)} funds from Pass 1")

Loaded 100 funds from Pass 1


In [3]:
PASS2_SYSTEM_PROMPT = """You are consolidating fund objective extractions that were made independently from multiple regulatory text columns for the same European mutual fund.

You will receive a JSON object where each key is a column name, and the value contains:
- "language": the language of that column
- "objectives": a list of objectives extracted from that column, each with:
  - "objective_text": verbatim text from the source
  - "objective_text_english": English translation
  - "objective_type": "financial" or "sustainable"

YOUR TASK:
1. MATCH equivalent objectives across columns/languages. The same objective may appear in English, French, German, Swedish, etc. These are duplicates and should be consolidated into ONE entry.
2. DEDUPLICATE: If multiple columns express the same objective (even in different words or languages), keep it only once.
3. For each unique objective, select the BEST English phrasing — prefer a native English source if available; otherwise use or improve the translation.
4. Classify each as "financial" or "sustainable".
5. Record which columns contained this objective (for traceability).

MATCHING GUIDANCE:
- "long-term capital growth" in English and "croissance du capital à long terme" in French = SAME objective
- "outperform the benchmark" and "exceed the benchmark index" = SAME objective (minor wording variation)
- "achieve capital growth" and "achieve capital growth and outperform the benchmark" — the second contains TWO objectives; match the first part and keep the second as separate
- Be generous in matching across languages but strict about not merging genuinely different objectives

OUTPUT FORMAT:
{
  "consolidated_objectives": [
    {
      "objective_number": 1,
      "objective_text_english": "the final English text for this objective",
      "objective_type": "financial" or "sustainable",
      "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", ...],
      "match_notes": "brief note on how columns were matched, or null if only found in one column"
    }
  ],
  "consolidation_notes": "any important notes about the consolidation process"
}

If Pass 1 found NO objectives in ANY column:
{
  "consolidated_objectives": [],
  "consolidation_notes": "NOT IDENTIFIED — no objectives found in any column"
}
"""

In [4]:
PASS2_FEW_SHOT = [
    {
        "fund_name": "Example Multilingual Fund",
        "pass1_data": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {"objective_text": "achieve capital growth", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "objective_text_english": "outperform the benchmark", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {"objective_text": "réaliser une croissance du capital", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "surperformer l'indice de référence", "objective_text_english": "outperform the benchmark index", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - German": {
                "language": "German",
                "objectives": [
                    {"objective_text": "Kapitalwachstum erzielen", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "die Benchmark übertreffen", "objective_text_english": "outperform the benchmark", "objective_type": "financial"},
                    {"objective_text": "Reduzierung der Treibhausgasemissionen", "objective_text_english": "reduction of greenhouse gas emissions", "objective_type": "sustainable"}
                ]
            }
        },
        "response": {
            "consolidated_objectives": [
                {
                    "objective_number": 1,
                    "objective_text_english": "achieve capital growth",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"],
                    "match_notes": "Same objective across English, French, and German columns"
                },
                {
                    "objective_number": 2,
                    "objective_text_english": "outperform the benchmark",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"],
                    "match_notes": "Same benchmark-beating objective across all three languages"
                },
                {
                    "objective_number": 3,
                    "objective_text_english": "reduction of greenhouse gas emissions",
                    "objective_type": "sustainable",
                    "found_in_columns": ["PRIIPS KID Objective - German"],
                    "match_notes": "Sustainability objective found only in German column"
                }
            ],
            "consolidation_notes": "Two financial objectives matched across all three languages. One sustainability objective found only in the German column."
        }
    }
]

In [5]:
def pass2_consolidate(fund_name, fund_id, pass1_data):
    """Consolidate per-column extractions into deduplicated objectives."""
    # Filter to only columns that had objectives
    cols_with_data = {}
    for col, data in pass1_data.items():
        if col.startswith('_'):
            continue
        if isinstance(data, dict) and 'objectives' in data and len(data['objectives']) > 0:
            cols_with_data[col] = data

    if not cols_with_data:
        return {
            "consolidated_objectives": [],
            "consolidation_notes": "NOT IDENTIFIED — Pass 1 found no objectives in any column"
        }

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

Pass 1 extractions (per-column):
{json.dumps(cols_with_data, indent=2)}"""

    messages = []
    for ex in PASS2_FEW_SHOT:
        messages.append({
            "role": "user",
            "content": f"Fund Name: {ex['fund_name']}\n\nPass 1 extractions (per-column):\n{json.dumps(ex['pass1_data'], indent=2)}"
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(ex["response"], indent=2)
        })

    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            temperature=0,
            system=PASS2_SYSTEM_PROMPT,
            messages=messages
        )
        print(f"   [{fund_name}] tokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

        text = response.content[0].text
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            if "```json" in text:
                return json.loads(text.split("```json")[1].split("```")[0].strip())
            elif "```" in text:
                return json.loads(text.split("```")[1].split("```")[0].strip())
            return {"_error": f"JSON parse error: {text[:300]}"}

    except Exception as e:
        print(f"   Error for {fund_name}: {e}")
        return {"_error": str(e)}

In [6]:
# === RUN PASS 2 ===
pass2_results = []

for idx in tqdm(range(len(p1_df)), desc="Pass 2 — Consolidate"):
    row = p1_df.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Fund_Name']
    pass1_data = row['pass1_raw']

    # Skip funds that errored in Pass 1
    if '_error' in pass1_data:
        pass2_results.append({
            'FundId': fund_id,
            'Fund_Name': fund_name,
            'pass2_raw': {'_error': f"Skipped — Pass 1 error: {pass1_data['_error']}"}
        })
        continue

    result = pass2_consolidate(fund_name, fund_id, pass1_data)

    pass2_results.append({
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'pass2_raw': result
    })

    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass2_df = pd.DataFrame(pass2_results)
print(f"\nPass 2 complete: {len(pass2_df)} funds processed")

Pass 2 — Consolidate:   1%|          | 1/100 [00:06<10:42,  6.49s/it]

   [MS INVF Global Brands Eq Inc Z] tokens — in: 3352, out: 433


Pass 2 — Consolidate:   2%|▏         | 2/100 [00:20<18:03, 11.05s/it]

   [DWS Global Value LD] tokens — in: 2947, out: 692


Pass 2 — Consolidate:   3%|▎         | 3/100 [00:24<12:08,  7.51s/it]

...

   [JB Edelweiss Swiss Equity SK Acc CHF] tokens — in: 2541, out: 312


Pass 2 — Consolidate:  98%|█████████▊| 98/100 [10:18<00:12,  6.17s/it]

   [WealthInv Qblue Bal GlbAkt AnsTran I] tokens — in: 1879, out: 284


Pass 2 — Consolidate: 100%|██████████| 100/100 [10:23<00:00,  6.24s/it]

   [Ethos Aktiefond A Utdelande (SEK)] tokens — in: 1944, out: 279

Pass 2 complete: 100 funds processed


In [7]:
# === FLATTEN FOR INSPECTION ===
flat_rows = []
for _, row in pass2_df.iterrows():
    raw = row['pass2_raw']
    base = {'FundId': row['FundId'], 'Fund_Name': row['Fund_Name']}

    if '_error' in raw:
        base['Number_of_Objectives'] = 0
        base['Consolidation_Notes'] = raw['_error']
        flat_rows.append(base)
        continue

    objs = raw.get('consolidated_objectives', [])
    base['Number_of_Objectives'] = len(objs)
    base['Consolidation_Notes'] = raw.get('consolidation_notes', '')

    for i in range(5):  # support up to 5 objectives
        if i < len(objs):
            o = objs[i]
            base[f'Objective_{i+1}'] = o.get('objective_text_english', '')
            base[f'Objective_{i+1}_Type'] = o.get('objective_type', '')
            base[f'Objective_{i+1}_Columns'] = ', '.join(o.get('found_in_columns', []))
        else:
            base[f'Objective_{i+1}'] = None
            base[f'Objective_{i+1}_Type'] = None
            base[f'Objective_{i+1}_Columns'] = None

    flat_rows.append(base)

pass2_flat = pd.DataFrame(flat_rows)

print("PASS 2 SUMMARY:")
print(f"  Funds: {len(pass2_flat)}")
print(f"  With ≥1 objective: {(pass2_flat['Number_of_Objectives'] > 0).sum()}")
print(f"  Avg objectives: {pass2_flat['Number_of_Objectives'].mean():.1f}")
print(f"\nObjective count distribution:")
print(pass2_flat['Number_of_Objectives'].value_counts().sort_index())

PASS 2 SUMMARY:
  Funds: 100
  With ≥1 objective: 94
  Avg objectives: 1.6

Objective count distribution:
Number_of_Objectives
0     6
1    49
2    27
3    11
4     6
5     1
Name: count, dtype: int64


In [8]:
# === SAVE PASS 2 OUTPUT ===
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Save flattened (human-readable) version
p2_flat_filename = f'Pass2_Consolidated_{len(pass2_flat)}_funds_{timestamp}.xlsx'
p2_flat_path = os.path.join(OUTPUT_DIR, p2_flat_filename)
pass2_flat.to_excel(p2_flat_path, index=False, engine='openpyxl')

# Save raw JSON version (for Pass 3 input)
p2_raw_df = pass2_df.copy()
p2_raw_df['pass2_raw'] = p2_raw_df['pass2_raw'].apply(json.dumps)
p2_raw_filename = f'Pass2_Raw_{len(pass2_df)}_funds_{timestamp}.xlsx'
p2_raw_path = os.path.join(OUTPUT_DIR, p2_raw_filename)
p2_raw_df.to_excel(p2_raw_path, index=False, engine='openpyxl')

print(f"Saved flattened: {p2_flat_filename}")
print(f"Saved raw JSON:  {p2_raw_filename}")
print(f"  → Use the raw JSON file as input to Pass 3")

Saved flattened: Pass2_Consolidated_100_funds_20260518_1335.xlsx
Saved raw JSON:  Pass2_Raw_100_funds_20260518_1335.xlsx
  → Use the raw JSON file as input to Pass 3
